# Multi-Head Attention (MHA)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个 head 的维度

        self.W_q = nn.Linear(d_model, d_model)  # Query 投影
        self.W_k = nn.Linear(d_model, d_model)  # Key 投影
        self.W_v = nn.Linear(d_model, d_model)  # Value 投影
        self.W_o = nn.Linear(d_model, d_model)  # 输出投影，融合所有 head

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B, T, _ = Q.shape                  # B: batch size, T: sequence length
        H, d_k = self.num_heads, self.d_k

        # project + split heads: (B, T, d_model) -> (B, H, T, d_k)
        Q = self.W_q(Q).view(B, T, H, d_k).transpose(1, 2)
        K = self.W_k(K).view(B, -1, H, d_k).transpose(1, 2)  # K/V 的 seq len 可以和 Q 不同
        V = self.W_v(V).view(B, -1, H, d_k).transpose(1, 2)

        # scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # 除以 sqrt(d_k) 防止点积过大导致 softmax 梯度消失
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))  # 被 mask 的位置设为 -inf，softmax 后概率为 0
        attn = F.softmax(scores, dim=-1)   # 对最后一维（key 方向）做 softmax
        out = attn @ V                     # 加权求和 Value，(B, H, T, d_k)

        # merge heads + project
        out = out.transpose(1, 2).contiguous().view(B, T, H * d_k)  # 拼回 (B, T, d_model)
        return self.W_o(out)

> 面试手写代码阶段，mask 可以不写，说一句「生产环境还需要加 causal mask 和 padding mask」就够了。

## view + transpose：split heads

```python
Q = self.W_q(Q).view(B, T, H, d_k).transpose(1, 2)
```

逐步拆解：

| 步骤 | 操作 | Shape |
|------|------|-------|
| 线性变换 | `self.W_q(Q)` | `(B, T, d_model)` |
| 拆 head | `.view(B, T, H, d_k)` | `(B, T, H, d_k)` |
| 调换维度 | `.transpose(1, 2)` | `(B, H, T, d_k)` |

`.transpose(1, 2)` 交换索引为 1 和 2 的两个维度，把 T 和 H 的位置对调。

**为什么要 transpose？** 后续 `@` 运算作用在最后两维 `(T, d_k)` 上，需要 H 在前面，才能让每个 head 并行独立地做 attention。

## masked_fill：把不该看的位置设为 -inf

```python
scores = scores.masked_fill(mask == 0, float('-inf'))
```

`mask == 0` 找出所有「不应该被 attention 的位置」，把那些位置的 score 设成负无穷。

原因：softmax 之后 $e^{-\infty} = 0$，所以这些位置的 attention 权重变成 0，模型完全忽略它们。

举个 causal mask 的例子：

```
mask:          scores 之前:        scores 之后:
1 0 0          0.5  0.3  0.1       0.5  -inf -inf
1 1 0    →     0.2  0.8  0.4   →   0.2   0.8 -inf
1 1 1          0.6  0.1  0.9       0.6   0.1  0.9
```

第一个位置只能看自己，第二个只能看前两个，第三个能看所有——这就是 causal mask 的效果。

## view 中的 -1：自动推断维度

```python
K = self.W_k(K).view(B, -1, H, d_k).transpose(1, 2)
```

Q 用的是 `view(B, T, H, d_k)`，因为 Q 的 seq_len 就是 T。

K/V 用 `-1` 而不是 `T`，是因为 cross-attention 场景下 K/V 的 seq_len 可以和 Q 不同，不能写死。`-1` 让 PyTorch 根据总元素数自动推断该维度：

```
总元素数 / (B * H * d_k) = seq_len_k
```

self-attention 时 `-1` 和 `T` 效果一样，写 `-1` 更通用，兼容 cross-attention。

## @ 对多维 tensor 的行为

```python
out = attn @ V
```

`@` / `torch.matmul` 的规则：**最后两维做矩阵乘法，前面所有维度当 batch 处理。**

```
attn: (B, H, T_q, T_k)
V:    (B, H, T_k, d_v)
→     (B, H, T_q, d_v)
```

B 和 H 自动并行，PyTorch 不需要任何额外指示，这是 `matmul` 的内置行为。

## 验证

In [ ]:
B, T, d_model, num_heads = 2, 10, 64, 8
x = torch.randn(B, T, d_model)

mha = MultiHeadAttention(d_model, num_heads)
out = mha(x, x, x)

print(f"input:  {x.shape}")
print(f"output: {out.shape}")  # 应为 (2, 10, 64)

## 与 nn.MultiheadAttention 对比 shape

In [ ]:
ref = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
ref_out, _ = ref(x, x, x)

print(f"my MHA output shape:  {out.shape}")
print(f"torch MHA output shape: {ref_out.shape}")
print(f"shape match: {out.shape == ref_out.shape}")

## Attention Weights

`nn.MultiheadAttention` 返回两个值，第二个是 attention weights，即 softmax 之后的矩阵：

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

shape 是 `(B, T, T)`，含义：

```
weights[b, i, j] = 第 b 个样本中，位置 i 对位置 j 的关注程度
```

比如 `T=4` 的句子，weights 长这样：

```
        位置0  位置1  位置2  位置3
位置0  [0.7,  0.1,  0.1,  0.1]   ← 位置0 主要关注自己
位置1  [0.2,  0.5,  0.2,  0.1]   ← 位置1 最关注自己
位置2  [0.1,  0.3,  0.4,  0.2]
位置3  [0.1,  0.1,  0.3,  0.5]
```

每行加起来 = 1（softmax 保证的）。

weights 主要用于**可视化**，帮助理解模型在关注哪些位置，推理时一般不需要，所以用 `_` 丢弃。

## 现代常见 Transformer 架构

**1. Encoder-only（双向）**
- 代表：BERT、RoBERTa
- 每个 token 能看全部上下文（无 causal mask）
- 适合：分类、NER、句子理解等理解类任务

**2. Decoder-only（单向）**
- 代表：GPT、LLaMA、Claude、Gemini
- 每个 token 只能看自己和之前的（causal mask）
- 适合：文本生成，是目前 LLM 的主流架构

**3. Encoder-Decoder（seq2seq）**
- 代表：T5、BART、原始 Transformer
- Encoder 双向理解输入，Decoder 单向生成输出，中间用 Cross-Attention 连接
- 适合：翻译、摘要等输入输出都是序列的任务

---

现在的趋势：**Decoder-only 统治 LLM。**

原因是 Decoder-only 用一个架构就能做所有任务（理解 + 生成），scaling 效果好，而 Encoder-only 和 Encoder-Decoder 的应用场景越来越窄。